# Baseline Instance Segmentation: YOLO11n-seg

Notebook ini mengunduh Panoptic Indoor Segmentation v5 dari Roboflow, melatih YOLO11n-seg, mengevaluasi model, dan menyimpan `best.pt`. Jalankan dengan Kaggle Accelerator **GPU**.

> Sebelum menjalankan: di Kaggle buka **Add-ons → Secrets**, tambahkan secret bernama `ROBOFLOW_API_KEY` dengan API key Roboflow-mu. Jangan menulis API key langsung di notebook.

Parameter baseline: 50 epoch, image size 640, batch 16. Sesuaikan batch jika GPU kehabisan memori.

In [1]:
!pip install -q ultralytics roboflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from pathlib import Path
from roboflow import Roboflow
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient
import yaml, torch

assert torch.cuda.is_available(), 'GPU belum aktif. Pilih GPU di Notebook Settings.'
print('GPU:', torch.cuda.get_device_name(0))
api_key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)
project = rf.workspace('panopticindoor').project('panoptic-indoor-segmentation')
version = project.version(5)
dataset = version.download('yolov11')
data_yaml = Path(dataset.location) / 'data.yaml'
print('Dataset:', dataset.location)
print(data_yaml.read_text())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to panoptic-indoor-segmentation-5 in yolov11:: 100%|██████████| 1856/1856 [00:00<00:00, 7589.10it/s]

Dataset: /kaggle/working/panoptic-indoor-segmentation-5
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 74
names: ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged', 'cardboard', 'ceiling-merged', 'cell phone', 'chair', 'clock', 'couch', 'counter', 'cup', 'curtain', 'dining table', 'door-stuff', 'fire-extinguisher', 'floor-other-merged', 'floor-wood', 'flower', 'food-other-merged', 'fork', 'hair drier', 'handbag', 'keyboard', 'knife', 'laptop', 'light', 'microwave', 'mirror-stuff', 'mouse', 'oven', 'pan', 'paper', 'paper-merged', 'person', 'pillow', 'plate', 'plug', 'potted plant', 'refrigerator', 'remote', 'rug-merged', 'scissor', 'scissors', 'shelf', 'sink', 'spoon', 'st-', 'stairs', 'stove-merged', 'table-merged', 'teddy bear', 'toaster', 'toilet', 'toothbrush', 'towel', 'toy', 'trash-bin', 'tree-merged', 'tv', 'vase', 'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood', 'wallet', 'window-blind', 'wind

## Periksa dataset
Pastikan `data.yaml` menunjukkan kelas yang benar serta path train/valid. Dataset ekspor juga sebaiknya memiliki split `test` agar evaluasi pada test set asli bisa dilakukan.

In [3]:
with open(data_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)
print('Classes:', data_cfg.get('names'))
print('Splits:', {k: data_cfg.get(k) for k in ['train', 'val', 'test']})

Classes: ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged', 'cardboard', 'ceiling-merged', 'cell phone', 'chair', 'clock', 'couch', 'counter', 'cup', 'curtain', 'dining table', 'door-stuff', 'fire-extinguisher', 'floor-other-merged', 'floor-wood', 'flower', 'food-other-merged', 'fork', 'hair drier', 'handbag', 'keyboard', 'knife', 'laptop', 'light', 'microwave', 'mirror-stuff', 'mouse', 'oven', 'pan', 'paper', 'paper-merged', 'person', 'pillow', 'plate', 'plug', 'potted plant', 'refrigerator', 'remote', 'rug-merged', 'scissor', 'scissors', 'shelf', 'sink', 'spoon', 'st-', 'stairs', 'stove-merged', 'table-merged', 'teddy bear', 'toaster', 'toilet', 'toothbrush', 'towel', 'toy', 'trash-bin', 'tree-merged', 'tv', 'vase', 'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood', 'wallet', 'window-blind', 'window-other']
Splits: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images'}


In [4]:
model = YOLO('yolo11n-seg.pt')
results = model.train(
    data=str(data_yaml),
    task='segment',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='yolo11n_seg_baseline',
    exist_ok=True,
    plots=True,
    seed=42
)
best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
print('Best weights:', best_weights)
print('Training log:', Path(results.save_dir) / 'results.csv')

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic-indoor-segmentation-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_seg_baseline, nbs=64

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


       2/50      4.35G      1.154      2.866      4.025       1.34          0        353        640: 100% ━━━━━━━━━━━━ 25/25 3.8it/s 6.6s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.3it/s 2.7s0.6s
                   all        166       2021      0.179      0.142      0.124     0.0971      0.204      0.144      0.127     0.0846

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/50      4.35G      1.061       2.52      3.444      1.288          0        336        640: 100% ━━━━━━━━━━━━ 25/25 3.9it/s 6.3s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.4it/s 2.6s0.6s
                   all        166       2021      0.366      0.158      0.131      0.101      0.377      0.156      0.134     0.0944

  

## Evaluasi
Selalu evaluasi pada split `test` bila tersedia. Jika tidak ada, notebook memakai `val` dan mencetak pemberitahuan; hasil val jangan dilaporkan sebagai hasil test.

In [5]:
eval_split = 'test' if data_cfg.get('test') else 'val'
if eval_split == 'val':
    print('PERHATIAN: split test tidak ditemukan; evaluasi ini memakai val.')
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(data_yaml), split=eval_split, imgsz=640, device=0, plots=True)
print('Evaluated split:', eval_split)
print('Box mAP50-95:', metrics.box.map)
print('Mask mAP50-95:', metrics.seg.map)
print('Mask mAP50:', metrics.seg.map50)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n-seg summary (fused): 114 layers, 2,867,206 parameters, 0 gradients, 9.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1550.9±1065.1 MB/s, size: 69.3 KB)
val: Scanning /kaggle/working/panoptic-indoor-segmentation-5/test/labels... 359 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 359/359 698.0it/s 0.5s0.0s
val: /kaggle/working/panoptic-indoor-segmentation-5/test/images/002_jpg.rf.941b44d4633817fd833de04127e2f3a3.jpg: 1 duplicate labels removed
val: /kaggle/working/panoptic-indoor-segmentation-5/test/images/236_jpeg.rf.ce43674993187cb554fcbd2c85146baf.jpg: 2 duplicate labels removed
val: New cache created: /kaggle/working/panoptic-indoor-segmentation-5/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.4it/s 6.8s0.2s
                   all        359       

In [7]:
# Lihat contoh prediksi mask dari gambar test (atau val jika test tidak tersedia).
split_path = data_cfg.get(eval_split)
dataset_root = Path(dataset.location)
# Roboflow biasanya menyimpan gambar di <split>/images; YAML bisa memakai path relatif berbeda.
split_dir = dataset_root / eval_split / 'images'
if not split_dir.exists() and split_path:
    yaml_split = Path(split_path)
    yaml_root = Path(data_cfg.get('path', data_yaml.parent))
    if not yaml_root.is_absolute():
        yaml_root = (data_yaml.parent / yaml_root).resolve()
    split_dir = yaml_split if yaml_split.is_absolute() else (yaml_root / yaml_split).resolve()
image_files = [p for p in split_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}] if split_dir.exists() else []
print('Images found:', len(image_files))
if image_files:
    best_model.predict(source=str(image_files[0]), conf=0.25, save=True, project='/kaggle/working/predictions', name='baseline', exist_ok=True)
    print('Prediction saved under /kaggle/working/predictions/baseline')

Images found: 359

image 1/1 /kaggle/working/panoptic-indoor-segmentation-5/test/images/151_jpg.rf.43a243a416f6cf6b2606472a4ca44a8d.jpg: 448x640 3 beds, 2 blankets, 2 ceiling-mergeds, 1 floor-other-merged, 5 pillows, 2 table-mergeds, 1 wall-other-merged, 11.7ms
Speed: 5.6ms preprocess, 11.7ms inference, 18.2ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /kaggle/working/predictions/baseline
Prediction saved under /kaggle/working/predictions/baseline


## Simpan dan unduh hasil
Di Kaggle, buka panel **Output** untuk mengunduh `best.pt`, `results.csv`, grafik training, dan folder hasil evaluasi. Simpan juga salinan notebook beserta log GPU untuk bukti proses tugas.

Bobot terbaik berada di `/kaggle/working/runs/segment/yolo11n_seg_baseline/weights/best.pt` (folder `runs` juga ditampilkan di output training).